# Coleta e preparacao das imagens de helipontos (z19 -> blocos 512)

Projeto P2 (CDIA - PUC-SP, Prof. Dr. Rooney Coelho). Notebook unico do trabalho inicial de dados:

1. baixa tiles XYZ do ESRI World Imagery no zoom 19 (cerca de 0,27 metros por pixel);
2. junta 2x2 tiles em blocos de 512x512 nativos (mesma ideia do notebook de apoio HIRES);
3. gera um world file por bloco (georreferencia em web mercator, semelhante ao .j2w das imagens t_orto);
4. mantem o split por bairro (treino_val e teste);
5. opcionalmente monta folhas de contato para acelerar a curadoria manual.

Os blocos ficam em 512x512. O redimensionamento para 640x640 e feito depois no Roboflow, junto da
anotacao (a propria ferramenta reescala e ajusta as caixas).

Atribuicao obrigatoria ao reproduzir as imagens: Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community

## Referencias das aulas e do enunciado

- Notebooks de apoio do projeto: Projeto_P2_Mosaico_Perdizes.ipynb (z=18) e Projeto_P2_Mosaico_Perdizes_HIRES.ipynb (z=19, blocos 512 a partir de 2x2 tiles). Base da coleta de tiles e da montagem 512.
- Aula de Localizacao, Deteccao e Segmentacao, notebook 05_formatos_e_ferramentas.ipynb: formato YOLO e organizacao de dataset.
- Aula de Localizacao, Deteccao e Segmentacao, notebook 03_yolo_e_metricas.ipynb: IoU, NMS e metricas usadas nas etapas seguintes do projeto.
- Sintese CDIA 2026, Bloco 1 (formatos de caixa) e Bloco 5 (dados, formatos e ferramentas).
- Enunciado P2: coleta via ESRI XYZ, zoom 19 para helipontos, resize 640x640 no Roboflow e curadoria manual obrigatoria.

## Configuracao

No Colab, monte o Drive se quiser persistir os arquivos (a sessao expira). Para apenas baixar e
depois subir no GitHub, pode deixar tudo em /content mesmo e usar a celula de download no final.

In [ ]:
# no colab descomente para instalar as dependencias
# !pip install -q requests pillow

import os
import math
import time
import shutil
import requests
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageDraw

# endpoint de tiles do esri world imagery e cabecalho de uso educacional
URL = ('https://server.arcgisonline.com/ArcGIS/rest/services/'
       'World_Imagery/MapServer/tile/{z}/{y}/{x}')
HEADERS = {'User-Agent': 'PUC-SP-AM-Aula-Educacional/1.0'}

ALVO = 'heliponto'
ZOOM = 19
ZOOM_ESPERADO_HELIPONTO = 19
TAMANHO_MINIMO_VALIDO = 2521
TAMANHO_TILE = 256
TAMANHO_BLOCO = 512
RAIO_TERRA = 6378137.0

PASTA_TILES = '/content/coleta_helipontos'
PASTA_BLOCOS = '/content/blocos_512'
PASTA_FOLHAS = '/content/folhas_contato'
PASTA_POSITIVOS = '/content/positivos'

# bbox no formato (lon_min, lat_min, lon_max, lat_max), igual ao enunciado
# papel define o destino no split por bairro: treino_val ou teste
REGIOES = {
    'itaim_faria_lima': {'bbox': (-46.694, -23.592, -46.672, -23.572), 'papel': 'treino_val'},
    'vila_olimpia': {'bbox': (-46.695, -23.604, -46.676, -23.590), 'papel': 'treino_val'},
    'brooklin_berrini': {'bbox': (-46.703, -23.624, -46.683, -23.606), 'papel': 'treino_val'},
    'avenida_paulista': {'bbox': (-46.660, -23.572, -46.640, -23.555), 'papel': 'teste'},
}

## Etapa 1 - Coleta dos tiles z19

Converte o bbox de cada bairro em tiles XYZ e baixa cada um. Baseado no script do enunciado e nos
notebooks de apoio Projeto_P2_Mosaico_Perdizes(_HIRES).

In [ ]:
def deg2tile(lat, lon, z):
    # converte latitude e longitude em graus para indice de tile x, y no zoom z
    n = 2.0 ** z
    x = int((lon + 180.0) / 360.0 * n)
    lat_rad = math.radians(lat)
    y = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def verificar_zoom(z, lat):
    # confere o zoom e imprime a resolucao no solo, como no notebook hires
    res_solo = 156543.03 * math.cos(math.radians(lat)) / (2.0 ** z)
    print('zoom usado:', z, '| resolucao no solo (m/px):', round(res_solo, 3))
    if ALVO == 'heliponto' and z != ZOOM_ESPERADO_HELIPONTO:
        print('atencao: para heliponto o zoom esperado e', ZOOM_ESPERADO_HELIPONTO)
    return res_solo

In [ ]:
def baixar_tile(z, x, y, pasta_saida):
    # baixa um unico tile com ate 3 tentativas e ignora placeholders pequenos
    nome = 'tile_z' + str(z) + '_x' + str(x) + '_y' + str(y) + '.jpg'
    destino = os.path.join(pasta_saida, nome)
    if os.path.exists(destino):
        return True
    tentativa = 0
    while tentativa < 3:
        try:
            resposta = requests.get(URL.format(z=z, x=x, y=y), headers=HEADERS, timeout=20)
            if resposta.status_code == 200 and len(resposta.content) > TAMANHO_MINIMO_VALIDO:
                with open(destino, 'wb') as arquivo:
                    arquivo.write(resposta.content)
                return True
        except requests.RequestException:
            pass
        tentativa = tentativa + 1
        time.sleep(1)
    # registra a falha indicando as coordenadas e continua sem interromper
    print('falha ao baixar tile z=' + str(z) + ' x=' + str(x) + ' y=' + str(y))
    return False


def listar_tiles(bbox, z):
    # converte o bbox em uma lista de coordenadas de tile a baixar
    lon_min, lat_min, lon_max, lat_max = bbox
    x_min, y_max = deg2tile(lat_min, lon_min, z)
    x_max, y_min = deg2tile(lat_max, lon_max, z)
    if x_min > x_max:
        x_min, x_max = x_max, x_min
    if y_min > y_max:
        y_min, y_max = y_max, y_min
    coordenadas = []
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            coordenadas.append((z, x, y))
    return coordenadas


def coletar_regiao(nome, bbox, papel, z):
    # baixa todos os tiles de uma regiao para a pasta do seu papel no split
    pasta_saida = os.path.join(PASTA_TILES, papel, nome)
    os.makedirs(pasta_saida, exist_ok=True)
    lon_min, lat_min, lon_max, lat_max = bbox
    lat_centro = (lat_min + lat_max) / 2.0
    print('coletando regiao', nome, 'papel', papel, 'bbox', bbox)
    verificar_zoom(z, lat_centro)
    coordenadas = listar_tiles(bbox, z)
    sucesso = 0
    with ThreadPoolExecutor(max_workers=4) as executor:
        tarefas = []
        for z_tile, x, y in coordenadas:
            tarefas.append(executor.submit(baixar_tile, z_tile, x, y, pasta_saida))
        for tarefa in tarefas:
            if tarefa.result():
                sucesso = sucesso + 1
    print('regiao', nome, 'tiles solicitados', len(coordenadas), 'tiles salvos', sucesso)
    return len(coordenadas), sucesso

In [ ]:
# executa a coleta de todas as regioes no zoom 19
os.makedirs(PASTA_TILES, exist_ok=True)
total_solicitado = 0
total_salvo = 0
for nome in REGIOES:
    bbox = REGIOES[nome]['bbox']
    papel = REGIOES[nome]['papel']
    solicitado, salvo = coletar_regiao(nome, bbox, papel, ZOOM)
    total_solicitado = total_solicitado + solicitado
    total_salvo = total_salvo + salvo
print('coleta concluida. total solicitado', total_solicitado, 'total salvo', total_salvo)
print('Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community')

## Etapa 2 - Montagem dos blocos 512x512

Junta 2x2 tiles z19 em um bloco 512x512 nativo (sem upscale), igual a comparacao do notebook HIRES.
Gera o world file de cada bloco e preserva o split por bairro.

In [ ]:
def parse_xy(nome):
    # extrai x e y do nome tile_z19_x12345_y67890.jpg
    base = nome
    if base.endswith('.jpg'):
        base = base[:-4]
    partes = base.split('_')
    x = int(partes[2][1:])
    y = int(partes[3][1:])
    return x, y


def escrever_world_file(caminho_imagem, z, bx, by, pixel):
    # world file em web mercator epsg 3857 a partir do tile superior esquerdo do bloco
    deslocamento_origem = 2.0 * math.pi * RAIO_TERRA / 2.0
    tamanho_tile_mercator = (2.0 * math.pi * RAIO_TERRA) / (2.0 ** z)
    canto_x = -deslocamento_origem + bx * tamanho_tile_mercator
    canto_y = deslocamento_origem - by * tamanho_tile_mercator
    centro_x = canto_x + pixel / 2.0
    centro_y = canto_y - pixel / 2.0
    caminho_world = caminho_imagem[:-4] + '.jgw'
    with open(caminho_world, 'w') as arquivo:
        print(pixel, file=arquivo)
        print(0, file=arquivo)
        print(0, file=arquivo)
        print(-pixel, file=arquivo)
        print(centro_x, file=arquivo)
        print(centro_y, file=arquivo)
    return caminho_world


def montar_blocos_regiao(pasta_regiao, pasta_saida):
    # agrupa os tiles em blocos 2x2 e salva blocos de 512 com world file
    os.makedirs(pasta_saida, exist_ok=True)
    tiles = {}
    for nome in os.listdir(pasta_regiao):
        if nome.endswith('.jpg'):
            x, y = parse_xy(nome)
            tiles[(x, y)] = os.path.join(pasta_regiao, nome)
    blocos = {}
    for chave in tiles:
        x, y = chave
        bx = x - (x % 2)
        by = y - (y % 2)
        blocos[(bx, by)] = True
    tamanho_tile_mercator = (2.0 * math.pi * RAIO_TERRA) / (2.0 ** ZOOM)
    pixel = tamanho_tile_mercator / TAMANHO_TILE
    salvos = 0
    for chave_bloco in blocos:
        bx, by = chave_bloco
        bloco = Image.new('RGB', (TAMANHO_BLOCO, TAMANHO_BLOCO))
        tem_algum = False
        for dx in range(2):
            for dy in range(2):
                chave_tile = (bx + dx, by + dy)
                if chave_tile in tiles:
                    imagem = Image.open(tiles[chave_tile]).convert('RGB')
                    bloco.paste(imagem, (dx * TAMANHO_TILE, dy * TAMANHO_TILE))
                    tem_algum = True
        if not tem_algum:
            continue
        nome_bloco = 'bloco_z' + str(ZOOM) + '_x' + str(bx) + '_y' + str(by) + '.jpg'
        caminho = os.path.join(pasta_saida, nome_bloco)
        bloco.save(caminho, quality=95)
        escrever_world_file(caminho, ZOOM, bx, by, pixel)
        salvos = salvos + 1
    return salvos

In [ ]:
# percorre papel/regiao e monta os blocos 512 preservando o split
total_blocos = 0
for papel in os.listdir(PASTA_TILES):
    pasta_papel = os.path.join(PASTA_TILES, papel)
    if not os.path.isdir(pasta_papel):
        continue
    for regiao in os.listdir(pasta_papel):
        pasta_regiao = os.path.join(pasta_papel, regiao)
        if not os.path.isdir(pasta_regiao):
            continue
        pasta_saida = os.path.join(PASTA_BLOCOS, papel, regiao)
        salvos = montar_blocos_regiao(pasta_regiao, pasta_saida)
        print('regiao', regiao, 'papel', papel, 'blocos salvos', salvos)
        total_blocos = total_blocos + salvos
print('total de blocos 512:', total_blocos)

## Etapa 3 (opcional) - Folhas de contato para a curadoria

A curadoria manual e obrigatoria na avaliacao. Para nao abrir bloco por bloco, esta etapa monta
grades de miniaturas numeradas. Voce escaneia as folhas, anota os numeros dos blocos com heliponto
e separa so esses. O criterio de presenca: pelo menos um heliponto (H branco em circulo) total ou
parcialmente visivel no bloco.

In [ ]:
def listar_blocos_ordenados(pasta_base):
    # percorre as subpastas e devolve os caminhos dos blocos em ordem estavel
    arquivos = []
    for raiz, pastas, nomes in os.walk(pasta_base):
        for nome in nomes:
            if nome.endswith('.jpg'):
                arquivos.append(os.path.join(raiz, nome))
    arquivos.sort()
    return arquivos


def construir_folhas(pasta_base, pasta_folhas, colunas, linhas, miniatura):
    # monta folhas de contato com miniaturas numeradas e salva o mapa indice -> caminho
    os.makedirs(pasta_folhas, exist_ok=True)
    arquivos = listar_blocos_ordenados(pasta_base)
    mapa = []
    por_folha = colunas * linhas
    indice = 0
    folha = 0
    while indice < len(arquivos):
        sheet = Image.new('RGB', (colunas * miniatura, linhas * miniatura), (0, 0, 0))
        desenho = ImageDraw.Draw(sheet)
        posicao = 0
        while posicao < por_folha and indice < len(arquivos):
            caminho = arquivos[indice]
            imagem = Image.open(caminho).convert('RGB').resize((miniatura, miniatura))
            coluna = posicao % colunas
            linha = posicao // colunas
            x = coluna * miniatura
            y = linha * miniatura
            sheet.paste(imagem, (x, y))
            desenho.text((x + 4, y + 4), str(indice), fill=(255, 255, 0))
            mapa.append((indice, caminho))
            indice = indice + 1
            posicao = posicao + 1
        sheet.save(os.path.join(pasta_folhas, 'folha_' + str(folha) + '.jpg'))
        folha = folha + 1
    with open(os.path.join(pasta_folhas, 'mapa_indices.csv'), 'w') as arquivo:
        arquivo.write('indice,caminho\n')
        for indice_bloco, caminho in mapa:
            arquivo.write(str(indice_bloco) + ',' + caminho + '\n')
    print('folhas geradas:', folha, '| blocos indexados:', len(mapa))
    return mapa


# gera as folhas de contato sobre os blocos 512
mapa = construir_folhas(PASTA_BLOCOS, PASTA_FOLHAS, 8, 8, 200)

In [ ]:
def separar_positivos(mapa, indices_positivos, pasta_base, pasta_destino):
    # copia os blocos positivos preservando a estrutura papel/regiao para manter o split
    dicionario = {}
    for indice_bloco, caminho in mapa:
        dicionario[indice_bloco] = caminho
    copiados = 0
    for indice in indices_positivos:
        if indice in dicionario:
            origem = dicionario[indice]
            relativo = os.path.relpath(origem, pasta_base)
            destino = os.path.join(pasta_destino, relativo)
            os.makedirs(os.path.dirname(destino), exist_ok=True)
            shutil.copy(origem, destino)
            world = origem[:-4] + '.jgw'
            if os.path.exists(world):
                shutil.copy(world, destino[:-4] + '.jgw')
            copiados = copiados + 1
    print('positivos copiados:', copiados, 'para', pasta_destino)


# preencha com os numeros das miniaturas que tem heliponto
indices_positivos = []

separar_positivos(mapa, indices_positivos, PASTA_BLOCOS, PASTA_POSITIVOS)

## Baixar para o PC

Gera um zip e baixa para o seu computador. Depois voce sobe no GitHub (GitHub Desktop ou Add file >
Upload files) e anota no Roboflow (la voce define o resize para 640x640, augmentation so no treino e
exporta no formato YOLOv8). Se ja tiver feito a triagem, zipe a pasta de positivos; senao, zipe os
blocos completos.

In [ ]:
# escolha o que zipar: positivos (apos triagem) ou todos os blocos
pasta_para_zipar = PASTA_POSITIVOS
if not os.path.exists(pasta_para_zipar) or len(os.listdir(pasta_para_zipar)) == 0:
    pasta_para_zipar = PASTA_BLOCOS
caminho_zip = shutil.make_archive('blocos_helipontos', 'zip', pasta_para_zipar)
print('arquivo zip gerado em:', caminho_zip)
try:
    from google.colab import files
    files.download(caminho_zip)
except Exception:
    print('fora do colab: o zip ja esta salvo no caminho acima')